In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import gymnasium as gym
# import matlab.engine
from typing import Optional, Union, Tuple, Any, Dict
import traci
import time, os, optparse, sys, random
import sumolib
from sumolib import checkBinary
import xlrd2



def xy2dist(path: np.ndarray) -> np.ndarray:
    """_summary_
    
    # 计算路径上每个点到起点的距离。

    参数:
    path (np.ndarray): 路径上的点坐标，形状为 (n, 2)，其中 n 是路径上的点的数量。

    返回:
    np.ndarray: 每个点到起点的距离，形状为 (n,)。
    """
    # 计算每个点到起点的距离
    dist = np.sqrt(np.sum(np.diff(path, axis=0)**2, axis=1))
    # 累加距离，得到每个点到起点的距离
    dist = np.insert(np.cumsum(dist), 0, 0)
    return dist


def can_reach_next_edge(veh_id):
    """
    # 判断该车当前车道能否到达下个目标edge
    :param veh_id: 主车ID
    :param next_target_edge: 目标下一个edge ID
    :return: bool，True=能到达，False=不能
    """
    # 步骤1：获取当前车道ID和下一个edgeID
    current_lane = traci.vehicle.getLaneID(veh_id)
    routeIndex = traci.vehicle.getRouteIndex(veh_id)
    route = traci.vehicle.getRoute(veh_id)
    if routeIndex == len(route) - 1:
        return True # 没有下一个边了，默认返回可达
    next_target_edge = route[routeIndex + 1]
    # 步骤2：获取该车道可驶出的edge列表
    outgoing_edges = traci.lane.getLinks(current_lane)
    # 步骤3：判断目标edge是否在列表中
    for i in outgoing_edges:
        if traci.lane.getEdgeID(i[0]) == next_target_edge:
            return True
    return False

def can_reach_next_edge_multi(veh_id, extend_lane_num=1):
    """
    # 判断该车当前车道和左右的N条车道能否到达下个目标edge
    :param veh_id: 主车ID
    :param extend_lane_num: 左右扩展车道数，默认1条, 即共 1 + 2*extend_lane_num 条车道
    :return: bool array，True=能到达，False=不能
    """
    lanes_to_judge = 1 + 2*extend_lane_num
    can_reach = np.zeros(lanes_to_judge, dtype=bool)
    # 步骤1：获取当前车道ID和下一个edgeID
    current_lane = traci.vehicle.getLaneID(veh_id)
    edge_id = traci.lane.getEdgeID(current_lane)
    lane_number = traci.edge.getLaneNumber(edge_id)
    current_lane_index = traci.vehicle.getLaneIndex(veh_id)
    right_rem_num = min(current_lane_index, extend_lane_num)
    left_rem_num = min(lane_number - 1 - current_lane_index, extend_lane_num)
    rem_indexs = np.arange(current_lane_index - right_rem_num, current_lane_index + 1 + left_rem_num)
    
    routeIndex = traci.vehicle.getRouteIndex(veh_id)
    route = traci.vehicle.getRoute(veh_id)
    if routeIndex == len(route) - 1:
        can_reach[rem_indexs] = True
        return np.flip(can_reach) # 没有下一个边了，默认返回可达
    
    next_target_edge = route[routeIndex + 1]
    # 步骤2：获取该车道可驶出的edge列表
    for i in rem_indexs:
        the_lane = edge_id + '_' + str(i)
        outgoing_edges = traci.lane.getLinks(the_lane)
        # 步骤3：判断目标edge是否在列表中
        for j in outgoing_edges:
            if traci.lane.getEdgeID(j[0]) == next_target_edge:
                can_reach[i - current_lane_index + extend_lane_num] = True
                break
    return np.flip(can_reach)


def calc_ttc(veh_id, clip=100.0, ego_speed=None, dummy_obs_dist=None):
    """_summary_
    # 计算给定车辆前/后分别的TTC
    Args:
        veh_id (string): 计算TTC的车辆ID
        clip (float, optional): TTC的clip值，默认100.0秒.
        ego_speed (float, optional): 主车速度，默认None. 
        dummy_obs_dist (float, optional): 虚拟障碍物距离，用于当前车道不能到达下一个edge时的TTC计算，默认None.
    """
    front_ttc = clip
    back_ttc = clip
    eps = 1e-6
    if ego_speed is None:
        ego_speed = traci.vehicle.getSpeed(veh_id)

    # 初始化前车和后车ID为None
    leader_id = None
    follower_id = None
    
    # 获取前车和后车ID
    leader = traci.vehicle.getLeader(veh_id)
    if leader is not None:
        leader_id, leader_dist = leader
    follower = traci.vehicle.getFollower(veh_id)
    if follower is not None:
        follower_id, follower_dist = follower
        
    # 如果存在前车就计算TTC
    if leader_id is not None:
        leader_speed = traci.vehicle.getSpeed(leader_id)
        front_ttc = leader_dist / (ego_speed - leader_speed + eps)
    if follower_id is not None:
        follower_speed = traci.vehicle.getSpeed(follower_id)
        back_ttc = follower_dist / (follower_speed - ego_speed + eps)
    
    if dummy_obs_dist is not None:
        front_ttc = min(front_ttc, dummy_obs_dist / (ego_speed + eps))
    
    return np.sign(front_ttc) * min(abs(front_ttc), clip), np.sign(back_ttc) * min(abs(back_ttc), clip)

def lane_change_state_encoder(veh_id):
    """_summary_
    # 手动编码车辆的换道状态
    Args:
        veh_id (string): 被编码车辆的ID

    Returns:
        lc_state (int array): 
                bit 0:是否处于紧急状态     urgent
                bit 1:是否被左前阻塞       blocked by left leader
                bit 2: 是否被左后阻塞      blocked by left follower
                bit 3: 是否被右前阻塞      blocked by right leader
                bit 4: 是否被右后阻塞      blocked by right follower
                bit 5: 是否重叠阻塞        overlapping
                bit 6: 是否正在向左换道    left
                bit 7: 是否正在向右换道    right
                bit 8: 是否可以向左换道    could left
                bit 9: 是否可以向右换道    could right
                bit 10: 是否正在战略变道   strategic
                bit 11: 是否在速度增益变道 speedGain
                bit 12: 是否在合作变道     cooperative
                bit 13: 是否子车道变换中   sublane
                bit 14: 有无TraCI指令     TraCI
                bit 15: 是否空间不足       insufficient space
                bit 16: 是否保持右车道     keepRight
    """
    # 初始化换道状态数组
    lc_state = np.zeros(17)
    
    # 分别获得左右换道状态并合并
    lc_state_L = traci.vehicle.getLaneChangeStatePretty(veh_id, traci.constants.LANECHANGE_LEFT)[1]  # 取考虑TraCI指令的状态[1]
    lc_state_R = traci.vehicle.getLaneChangeStatePretty(veh_id, traci.constants.LANECHANGE_RIGHT)[1]
    state_fdbk = tuple(dict.fromkeys(list(lc_state_L) + list(lc_state_R)).keys()) # 合并并去重
    
    # 检查是否可以向左右换道
    could_lc_L = traci.vehicle.couldChangeLane(veh_id, traci.constants.LANECHANGE_LEFT)
    could_lc_R = traci.vehicle.couldChangeLane(veh_id, traci.constants.LANECHANGE_RIGHT)
    
    lc_state[0] = 1 if 'urgent' in state_fdbk else 0
    lc_state[1] = 1 if 'blocked by left leader' in state_fdbk else 0
    lc_state[2] = 1 if 'blocked by left follower' in state_fdbk else 0
    lc_state[3] = 1 if 'blocked by right leader' in state_fdbk else 0
    lc_state[4] = 1 if 'blocked by right follower' in state_fdbk else 0
    lc_state[5] = 1 if 'overlapping' in state_fdbk else 0
    lc_state[6] = 1 if 'left' in state_fdbk else 0
    lc_state[7] = 1 if 'right' in state_fdbk else 0
    lc_state[8] = 1 if could_lc_L else 0
    lc_state[9] = 1 if could_lc_R else 0
    lc_state[10] = 1 if 'strategic' in state_fdbk else 0
    lc_state[11] = 1 if 'speedGain' in state_fdbk else 0
    lc_state[12] = 1 if 'cooperative' in state_fdbk else 0
    lc_state[13] = 1 if 'sublane' in state_fdbk else 0
    lc_state[14] = 1 if 'TraCI' in state_fdbk else 0
    lc_state[15] = 1 if 'insufficient space' in state_fdbk else 0
    lc_state[16] = 1 if 'keepRight' in state_fdbk else 0
    
    return lc_state
    



(100.0, -458.7786005357889)

In [ ]:
leader = traci.vehicle.getLeader('t_0')
if leader is not None:
    leader_id, leader_dist = leader
print(leader_id, leader_dist)

flow_main.42 61.27820329452052


In [ ]:
# 检测是否已经添加环境变量
if 'SUMO_HOME' in os.environ:
    tools = os.path.join(os.environ['SUMO_HOME'], 'tools')
    sys.path.append(tools)
else:
    sys.exit("please declare environment variable 'SUMO_HOME'")

# sumo自带的，检查gui版本的sumo是否存在
def get_options():
    optParser = optparse.OptionParser()
    optParser.add_option("--nogui", action="store_true",
                         default=False, help="run the commandline version of sumo")
    options, args = optParser.parse_args()
    return options

In [ ]:
traci.simulationStep(155.)
traci.vehicle.getRouteIndex('t_0')

0

: 

In [ ]:

# sumocfgfile = os.path.join(os.getcwd(), "..", "..", "..", "data", "test_cases", "no1_4500_pass_1.sumocfg")
sumocfgfile = os.path.join(os.getcwd(), "..", "..", "..", "data", "test_cases", "no14_3000_merge_4.sumocfg")

# 检查文件是否存在
if os.path.exists(sumocfgfile):
    print("文件存在")
else:
    print("文件不存在")
    

# for _ in range(5):
#     traci.start(['sumo', '-c', sumocfgfile, '--start', '--quit-on-end','--seed',str(_)])  # 打开sumocfg文件
#     for step in range(0,1):
#         traci.simulationStep(step + 1.)  # 一步一步（一帧一帧）进行仿真
#         print(f'step: {step + 1}')
#         veh_list = traci.vehicle.getIDList()
#         print(traci.vehicle.getAccel(veh_list[0]))
        
        
#     traci.close()
traci.start(['sumo-gui', '-c', sumocfgfile, '--start', '--quit-on-end','--seed',str(1000)])  # 打开sumocfg文件

ViewID = 'View #0'
schema = 'real world'
egoID = 't_0'
traci.simulation.getTime()
traci.gui.setSchema(ViewID, schema)
traci.gui.setZoom(ViewID, 80000)
traci.gui.trackVehicle(ViewID, egoID)

# simTime = 185.
# traci.simulationStep(185.)

文件存在


* **自车状态**： 

    - 速度
    - **期望速度**
    - （相对）位置
    - 位于匝道/主路（one-hot）
    - 前/后/左前/左后/右前/右后TTC
    - 换道状态（one-hot）
    - 车道剩余距离
    - 意图（one-hot）
    - **可通行车道mask**
    - **上次换道间隔**
    - 当前限速
    - 最高速度
    - 车辆类型（可学习嵌入）

* **他车状态**： 

    - 速度
    - （相对）位置
    - 位于匝道/主路（one-hot）
    - 前/后/左前/左后/右前/右后TTC
    - 换道状态（one-hot）
    - 车道剩余距离
    - 意图（one-hot）
    - 当前限速
    - 最高速度
    - 车辆类型（可学习嵌入）

* **奖励函数**： 
   - 安全（TTC、MEI）✅️
   - 效率（和限速的SoothL1）✅️
   - 导航（处于符合导航要求的可通行车道）✅️
   - 规则（限速、靠右行驶）✅️
   - 舒适/节能（加速度平方）✅️
   - 操作合规（过低/高车速指令、违规换道）✅️
    - 违规换道：
        - 正在换道时，再次下发指令扣分
        - 换道指令下发后，出现阻塞状态扣分


In [ ]:
traci.close()

In [ ]:
lc_called = 0
for i in range(100):
    simTime += 1.
    traci.simulationStep(simTime)
    print('LC state L        : ',traci.vehicle.getLaneChangeStatePretty(egoID, traci.constants.LANECHANGE_LEFT))
    print('LC state R        : ',traci.vehicle.getLaneChangeStatePretty(egoID, traci.constants.LANECHANGE_RIGHT))
    if simTime % 10 == 0:
        traci.vehicle.changeLaneRelative(egoID, (-1)**lc_called, 4)
        print('change lane to ', (-1)**lc_called)
        lc_called += 1

LC state L        :  ((), ())
LC state R        :  ((), ())
LC state L        :  ((), ())
LC state R        :  ((), ())
LC state L        :  ((), ())
LC state R        :  ((), ())
LC state L        :  ((), ())
LC state R        :  ((), ())
LC state L        :  ((), ())
LC state R        :  ((), ())
change lane to  1
LC state L        :  ((), ())
LC state R        :  ((), ())
LC state L        :  ((), ())
LC state R        :  ((), ())
LC state L        :  ((), ())
LC state R        :  ((), ())
LC state L        :  ((), ())
LC state R        :  ((), ())
LC state L        :  ((), ())
LC state R        :  ((), ())
LC state L        :  ((), ())
LC state R        :  ((), ())
LC state L        :  ((), ())
LC state R        :  ((), ())
LC state L        :  ((), ())
LC state R        :  ((), ())
LC state L        :  ((), ())
LC state R        :  ((), ())
LC state L        :  ((), ())
LC state R        :  ((), ())
change lane to  -1
LC state L        :  ((), ())
LC state R        :  ((), ())
LC 

In [ ]:
left_lc_stat = traci.vehicle.getLaneChangeStatePretty(egoID, traci.constants.LANECHANGE_LEFT)
print('right' in left_lc_stat[0])


True


In [ ]:
lc_state_L = traci.vehicle.getLaneChangeStatePretty('t_0', traci.constants.LANECHANGE_LEFT)[1]
lc_state_R = traci.vehicle.getLaneChangeStatePretty('t_0', traci.constants.LANECHANGE_RIGHT)[1]
tuple(dict.fromkeys(list(lc_state_L) + list(lc_state_R)).keys())


('left',
 'TraCI',
 'urgent',
 'blocked by left follower',
 'overlapping',
 'sublane')

In [ ]:
"""_summary_
        % 车道变换模型区分了四种变换车道的原因：
    % 
    % 战略性（变道以继续行驶路线）
    % 合作性（为允许其他车辆变道而进行的变道）
    % 提速（其他车道允许更快行驶）
    % 靠右行驶的义务
    % 在每个模拟步骤中，车道变换模型会计算一个内部请求，以决定变换车道还是保持在当前车道。
    % 
    % 如果外部车道变换指令（0x13）与内部请求存在冲突，将通过车辆车道变换模式的当前值来解决。给定的整数被解释为一个位集（bit0是最低有效位），包含以下字段：
    % 
    % bit1、bit0：00 = 不做策略变更；01 = 若与TraCI请求不冲突，则进行策略变更；10 = 即使要推翻TraCI请求，也要进行策略变更
    % bit3、bit2：00 = 不进行协作更改；01 = 若与TraCI请求不冲突，则进行协作更改；10 = 即使覆盖TraCI请求，也要进行协作更改
    % bit5、bit4：00 = 不进行速度增益更改；01 = 若与TraCI请求不冲突，则进行速度增益更改；10 = 即使要覆盖TraCI请求，也要进行速度增益更改
    % bit7、bit6：00 = 不进行右车道变更；01 = 若与TraCI请求不冲突，则进行右车道变更；10 = 即使要覆盖TraCI请求，也要进行右车道变更
    % bit9，bit8： 00 = 遵循TraCI请求时不考虑其他驾驶员，调整速度以完成请求 01 = 遵循TraCI请求时避免即时碰撞，调整速度以完成请求 10 = 变道时考虑其他车辆的速度/制动间隙，调整速度以完成请求 11 = 变道时考虑其他车辆的速度/制动间隙，不调整速度
    % 00 = 遵循TraCI请求时不考虑其他驾驶员，调整速度以满足请求
    % 01 = 遵循TraCI请求时避免即时碰撞，调整速度以完成请求
    % 10 = 变道时尊重其他车辆的速度/刹车距离，调整速度以满足请求
    % 11 = 变道时尊重其他车辆的速度/制动间距，不进行速度调整
    % bit11、bit10：00 = 不进行子车道变更；01 = 若与TraCI请求不冲突，则进行子车道变更；10 = 即使要覆盖TraCI请求，也要进行子车道变更

    if params.manual_control
        traci.vehicle.setLaneChangeMode(ego.vehID, 0b000100010010); % bit11 10 ... 2 1 0  % 我们控制的到时候的模式
    else
        traci.vehicle.setLaneChangeMode(ego.vehID, 0b011001010110); % bit11 10 ... 2 1 0  % 自己采集数据的时候的模式
    end
    % traci.vehicle.setLaneChangeMode(ego.vehID, 256); % 限制不让车辆自主换道，但是可以sublanechange

    % setSpeedMode 说明
    % bit0: Regard safe speed
    % bit1: Regard maximum acceleration
    % bit2: Regard maximum deceleration
    % bit3: Regard right of way at intersections (only applies to approaching foe vehicles outside the intersection)
    % bit4: Brake hard to avoid passing a red light
    % bit5: Disregard right of way within intersections (only applies to foe vehicles that have entered the intersection).
    % bit6: Disregard speed limit.

    traci.vehicle.setSpeedMode(ego.vehID,0b0011111); % bit6 bit5 bit4 ... bit0. bit0是考虑安全速度
    
"""

print('Lane Change Mode: ',bin(traci.vehicle.getLaneChangeMode(egoID)))
print('Speed Mode: ',bin(traci.vehicle.getSpeedMode(egoID)))

Lane Change Mode:  0b11001010101
Speed Mode:  0b11111


In [ ]:
simTime += 1.
extend_lane_num = 3
traci.simulationStep(simTime)
speed = traci.vehicle.getSpeed(egoID)
speed_lim = traci.vehicle.getAllowedSpeed(egoID)
laneID = traci.vehicle.getLaneID(egoID)
edgeID = traci.lane.getEdgeID(laneID)
laneLength = traci.lane.getLength(laneID)
lanePosition = traci.vehicle.getLanePosition(egoID)
remLaneDist = max(0, laneLength - lanePosition)
can_reach_multi = can_reach_next_edge_multi(egoID,extend_lane_num=extend_lane_num)
can_reach = can_reach_multi[extend_lane_num]
dummy_obs_dist = remLaneDist if not can_reach else 1e3
front_ttc, back_ttc = calc_ttc(egoID, ego_speed=speed, dummy_obs_dist=dummy_obs_dist)
lc_state = lane_change_state_encoder(egoID)

# 打印can_reach_multi(false:●, true:○)
print('can_reach_multi   : ', ''.join(['○' if i else '●' for i in can_reach_multi]))
print('front / back TTC  : {:.2f} / {:.2f} s'.format(front_ttc, back_ttc))
print('speed / speedlim  : {:.2f} / {:.2f} km/h'.format(speed*3.6, speed_lim*3.6))
print('laneID / edgeID   : ',laneID,',', edgeID)
print('lanePos / remDist : {:.2f} / {:.2f} m'.format(lanePosition, remLaneDist))
print('leader            : ',traci.vehicle.getLeader(egoID))
print('follower          : ',traci.vehicle.getFollower(egoID))
print('left leaders      : ',traci.vehicle.getLeftLeaders(egoID))
print('left followers    : ',traci.vehicle.getLeftFollowers(egoID))
print('right leaders     : ',traci.vehicle.getRightLeaders(egoID))
print('right followers   : ',traci.vehicle.getRightFollowers(egoID))
print('could LC left     : ',traci.vehicle.couldChangeLane(egoID, traci.constants.LANECHANGE_LEFT))
print('could LC right    : ',traci.vehicle.couldChangeLane(egoID, traci.constants.LANECHANGE_RIGHT))
print('LC state L        : ',traci.vehicle.getLaneChangeStatePretty(egoID, traci.constants.LANECHANGE_LEFT))
print('LC state R        : ',traci.vehicle.getLaneChangeStatePretty(egoID, traci.constants.LANECHANGE_RIGHT))
print('LC state          : ', ''.join(['○' if i else '●' for i in lc_state]))


can_reach_multi   :  ●○○○●●●
front / back TTC  : -100.00 / -100.00 s
speed / speedlim  : 75.60 / 79.44 km/h
laneID / edgeID   :  413494440#2_0 , 413494440#2
lanePos / remDist : 3726.07 / 7138.82 m
leader            :  ('flow_main.42', 142.349364867423)
follower          :  ('flow_main.49', 36.853939023954354)
left leaders      :  (('flow_main.37', 185.32538008519487), ('flow_main.61', -4.207010823705787), ('flow_main.58', 142.42924852380884), ('flow_main.37', 185.32538008519487), ('flow_main.51', 257.39055580252534))
left followers    :  (('flow_main.95', 1776.087237705242), ('flow_main.47', 159.4253018143827), ('flow_main.76', 15.198272555713174), ('flow_main.77', 84.34503736292118), ('flow_merge.51', 2443.8566718278653))
right leaders     :  ()
right followers   :  ()
could LC left     :  True
could LC right    :  False
LC state L        :  (('stay', 'sublane'), ('stay', 'sublane'))
LC state R        :  ((), ())
LC state          :  ●●●●●●●●○●●●●○●●●


In [ ]:
routes = traci.vehicle.getRoute(egoID)
print(routes)


('479189099#1-AddedOffRampEdge', '135267611', '413494440#2-AddedOnRampEdge', '413494440#2', '413494440#2-AddedOffRampEdge')


In [ ]:
traci.vehicle.changeLaneRelative(egoID, 1, 4)

In [ ]:
xy2dist(np.array(traci.lane.getShape('135267611_0')))

array([  0.        ,  35.59655039,  48.74025611,  68.67189177,
        89.40714918, 110.22710355, 129.55718374, 146.99193639,
       163.78223419, 180.19950558, 197.69335021, 218.09353894,
       236.03454504, 258.44076069, 279.96575314, 308.00544287,
       336.34099298, 468.98592127])

In [ ]:
traci.vehicle.changeLaneRelative(egoID, -1, 1)
# 转换为2进制
bin(traci.vehicle.getLaneChangeMode(egoID))

'0b11001010101'

In [ ]:
from compass_env.envs.compass_highway_env import CompassHighwayEnv

ModuleNotFoundError: No module named 'compass_env'

In [ ]:
traci.close()

In [ ]:
import numpy as np
import gymnasium as gym
from gymnasium import spaces


class CompassHighwayEnv(gym.Env):
    """
    Custom Environment that follows gym interface.
    This is a wrapped Environment 
    """

    # Because of google colab, we cannot implement the GUI ('human' render mode)
    metadata = {"render_modes": ["console", "human"]}

    # Define constants for clearer code
    LEFT = 0
    RIGHT = 1

    def __init__(self, grid_size=10, render_mode="console"):
        super(CompassHighwayEnv, self).__init__()
        self.render_mode = render_mode
        # 检测是否已经添加环境变量
        if 'SUMO_HOME' in os.environ:
            tools = os.path.join(os.environ['SUMO_HOME'], 'tools')
            sys.path.append(tools)
        else:
            sys.exit("please declare environment variable 'SUMO_HOME'")
        self.sumoCfgFile = os.path.join(os.getcwd(), "..", "..", "..", "data", "test_cases", "no1_4500_pass_1.sumocfg")
        # 检查文件是否存在
        if os.path.exists(self.sumoCfgFile):
            print("仿真文件存在~")
        else:
            print("仿真文件不存在！")
        
        if render_mode == "human":
            render_cmd = 'sumo-gui'
        else:
            render_cmd = 'sumo'
        self.render_cmd = render_cmd
        
        
       
 
        # Initialize the agent at the right of the grid
        self.agent_pos = grid_size - 1

        # Define action and observation space
        # They must be gym.spaces objects
        # Example when using discrete actions, we have two: left and right
        n_actions = 2
        self.action_space = spaces.Discrete(n_actions)
        # The observation will be the coordinate of the agent
        # this can be described both by Discrete and Box space
        self.observation_space = spaces.Box(
            low=0, high=self.grid_size, shape=(1,), dtype=np.float32
        )

    def reset(self, seed=None, options=None):
        """
        Important: the observation must be a numpy array
        :return: (np.array)
        """
        super().reset(seed=seed, options=options)
        
        if seed is None:
            seed = np.random.randint(0, 10000)
        try:
            traci.close() # 如果环境还没关闭，就手动关闭
        except:
            pass
        # 启动SUMO仿真
        traci.start([self.render_cmd, '-c', self.sumoCfgFile, '--start', '--quit-on-end','--seed',str(seed)])  # 打开sumocfg文件
        # Initialize the agent at the right of the grid
        self.agent_pos = self.grid_size - 1
        # here we convert to float32 to make it more general (in case we want to use continuous actions)
        return np.array([self.agent_pos]).astype(np.float32), {}  # empty info dict

    def step(self, action):
        if action == self.LEFT:
            self.agent_pos -= 1
        elif action == self.RIGHT:
            self.agent_pos += 1
        else:
            raise ValueError(
                f"Received invalid action={action} which is not part of the action space"
            )

        # Account for the boundaries of the grid
        self.agent_pos = np.clip(self.agent_pos, 0, self.grid_size)

        # Are we at the left of the grid?
        terminated = bool(self.agent_pos == 0)
        truncated = False  # we do not limit the number of steps here

        # Null reward everywhere except when reaching the goal (left of the grid)
        reward = 1 if self.agent_pos == 0 else 0

        # Optionally we can pass additional info, we are not using that for now
        info = {}

        return (
            np.array([self.agent_pos]).astype(np.float32), 
            reward,                                        
            terminated,                                    
            truncated,                                      
            info,
        )

    def render(self):
        # agent is represented as a cross, rest as a dot
        pass

    def close(self):
        try:
            traci.close() 
        except:
            pass
    
    


    